# PropertyLens Feature Engineering (11-factor coverage)

This notebook migrates feature-building logic into `02_feature_layer` and produces a training-ready feature table for HDB resale price prediction.

Covered factors:
1. level
2. lease remaining years
3. size
4. room count
5. distance to MRT
6. orientation / facing road score
7. distance to highway
8. distance to foodcourt
9. large commercial access
10. all-school proximity (distance + density)
11. primary-school tier score within 1km (implicit quality from enrollment competition)

In [1]:
import hashlib
import json
import math
import os
import re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.neighbors import BallTree

SEED = 42
np.random.seed(SEED)

cwd = Path.cwd()
ROOT = cwd if (cwd / 'hf_data').exists() else cwd.parent  # works from repo root or any sublayer
RAW_DIR = ROOT / "01_data_layer" / "raw"
GEO_DIR = RAW_DIR / "google_geo"
SCHOOL_DIR = RAW_DIR / "schools"
BIZ_DIR = RAW_DIR / "SoldandRentedHDBPropertiesandFacilities"
FEATURE_DIR = ROOT / "02_feature_layer" / "training"
OUTPUT_DIR = FEATURE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_DATE = datetime.now().strftime("%Y%m%d")


def latest_file_by_pattern(directory, pattern):
    """Return the most recently dated file matching *pattern* in *directory*, or None."""
    candidates = sorted(Path(directory).glob(pattern))
    return candidates[-1] if candidates else None


print("ROOT:", ROOT)
print("RAW_DIR:", RAW_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("BIZ_DIR:", BIZ_DIR)

ROOT: /Users/lorenzolou/VScode/PropertyLens
RAW_DIR: /Users/lorenzolou/VScode/PropertyLens/01_data_layer/raw
OUTPUT_DIR: /Users/lorenzolou/VScode/PropertyLens/02_feature_layer/training/outputs
BIZ_DIR: /Users/lorenzolou/VScode/PropertyLens/01_data_layer/raw/SoldandRentedHDBPropertiesandFacilities


In [2]:
def load_hdb_2015_plus() -> pd.DataFrame:
    hdb_dir = RAW_DIR / 'ResaleFlatPrices'
    files = sorted(hdb_dir.glob('*.csv'))
    if not files:
        raise FileNotFoundError(f'No HDB CSV found in {hdb_dir}')

    # 🔧 FIX: Exclude backup files to prevent ~9x data duplication
    # Problem: Directory contains backup files like "...From Jan 2015 to Dec 2016_backup_20260403.csv"
    # Solution: Load only primary files (no "_backup" in filename)
    files = [f for f in files if 'backup' not in f.name.lower()]
    
    if not files:
        raise FileNotFoundError(f'No primary HDB CSV found in {hdb_dir} (only backups exist)')

    print(f'Loading {len(files)} HDB file(s) from ResaleFlatPrices/')
    
    parts = []
    for fp in files:
        print(f'  → {fp.name}')
        d = pd.read_csv(fp)
        d['month_dt'] = pd.to_datetime(d['month'], errors='coerce')
        d = d[d['month_dt'].dt.year >= 2015].copy()
        d['source_file'] = fp.name
        parts.append(d)

    hdb = pd.concat(parts, ignore_index=True)
    hdb['address_key'] = (
        hdb['block'].astype(str).str.strip() + ' ' + hdb['street_name'].astype(str).str.strip()
    ).str.upper()
    hdb['transaction_year'] = hdb['month_dt'].dt.year

    # Use the same source_id convention as raw collection notebook for stable joining
    hdb['source_id'] = hdb.apply(
        lambda r: hashlib.md5(
            f"{str(r['block']).strip()}|{str(r['street_name']).strip()}|{str(r['town']).strip()}".encode('utf-8')
        ).hexdigest(),
        axis=1,
    )

    hdb['resale_price'] = pd.to_numeric(hdb['resale_price'], errors='coerce')
    hdb['floor_area_sqm'] = pd.to_numeric(hdb['floor_area_sqm'], errors='coerce')
    hdb['lease_commence_date'] = pd.to_numeric(hdb['lease_commence_date'], errors='coerce')

    # Factor 1: level (midpoint of storey range)
    level_bounds = hdb['storey_range'].astype(str).str.extract(r'(\d+)\s+TO\s+(\d+)')
    hdb['level_mid'] = (
        pd.to_numeric(level_bounds[0], errors='coerce') + pd.to_numeric(level_bounds[1], errors='coerce')
    ) / 2

    # Factor 2: lease remaining years, assuming 99-year lease
    hdb['lease_remaining_years'] = 99 - (hdb['transaction_year'] - hdb['lease_commence_date'])
    hdb['lease_remaining_years'] = hdb['lease_remaining_years'].clip(lower=0, upper=99)

    # Factor 4: room count from flat_type
    room_num = hdb['flat_type'].astype(str).str.extract(r'(\d+)')
    hdb['room_count'] = pd.to_numeric(room_num[0], errors='coerce')
    hdb.loc[hdb['flat_type'].astype(str).str.contains('EXECUTIVE', case=False, na=False), 'room_count'] = 5
    hdb.loc[hdb['flat_type'].astype(str).str.contains('MULTI', case=False, na=False), 'room_count'] = 6

    return hdb

In [3]:
def normalize_address_from_requested(series: pd.Series) -> pd.Series:
    """Strip ', Singapore' suffix so requested_address matches address_key format."""
    return (
        series.astype(str)
        .str.replace(r",\s*Singapore\s*$", "", regex=True)
        .str.strip()
        .str.upper()
    )

hdb = load_hdb_2015_plus()
print(f"Loaded {len(hdb):,} HDB transactions")
hdb.head(2)

Loading 3 HDB file(s) from ResaleFlatPrices/
  → Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv
  → Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv
  → Resale flat prices based on registration date from Jan-2017 onwards.csv


Loaded 263,004 HDB transactions


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,month_dt,source_file,address_key,transaction_year,source_id,level_mid,lease_remaining_years,room_count
0,2015-01,ANG MO KIO,3 ROOM,174,ANG MO KIO AVE 4,07 TO 09,60.0,Improved,1986,70.0,255000.0,2015-01-01,Resale Flat Prices (Based on Registration Date...,174 ANG MO KIO AVE 4,2015,f10a0ad9f92b5fed1af5d220a2ca1e57,8.0,70,3.0
1,2015-01,ANG MO KIO,3 ROOM,541,ANG MO KIO AVE 10,01 TO 03,68.0,New Generation,1981,65.0,275000.0,2015-01-01,Resale Flat Prices (Based on Registration Date...,541 ANG MO KIO AVE 10,2015,5ba09c4a5cb55a06441120282b2df0de,2.0,65,3.0


In [4]:
# Load geocode/accessibility features prepared in data layer
geo_acc_fp = latest_file_by_pattern(GEO_DIR, 'hdb_geo_accessibility_noise_features_*.csv')
geo_hw_fp = latest_file_by_pattern(GEO_DIR, 'onemap_hdb_geocode_with_highway_dist_*.csv')

if geo_acc_fp is None:
    raise FileNotFoundError('Missing hdb_geo_accessibility_noise_features_*.csv in raw/google_geo')
if geo_hw_fp is None:
    raise FileNotFoundError('Missing onemap_hdb_geocode_with_highway_dist_*.csv in raw/google_geo')

geo_acc = pd.read_csv(geo_acc_fp)
geo_hw = pd.read_csv(geo_hw_fp)

# Primary join path: source_id (stable key from raw collection)
acc_keep = [
    'source_id',
    'lat', 'lng',
    'nearest_mrt_km',
    'road_noise_score',
    'facing_road_noise_proxy',
]
acc_keep = [c for c in acc_keep if c in geo_acc.columns]
geo_acc_slim = geo_acc[acc_keep].drop_duplicates('source_id')

hw_keep = ['source_id', 'highway_distance_km']
hw_keep = [c for c in hw_keep if c in geo_hw.columns]
geo_hw_slim = geo_hw[hw_keep].drop_duplicates('source_id')

feat = hdb.merge(geo_acc_slim, on='source_id', how='left').merge(geo_hw_slim, on='source_id', how='left')

# Fallback join for any unmatched rows via normalized address key
missing_mask = feat['lat'].isna() | feat['lng'].isna()
if missing_mask.any():
    geo_acc_addr = geo_acc.copy()
    geo_hw_addr = geo_hw.copy()
    geo_acc_addr['address_key'] = normalize_address_from_requested(geo_acc_addr['requested_address'])
    geo_hw_addr['address_key'] = normalize_address_from_requested(geo_hw_addr['requested_address'])

    geo_acc_addr = geo_acc_addr[['address_key', 'lat', 'lng', 'nearest_mrt_km', 'road_noise_score', 'facing_road_noise_proxy']].drop_duplicates('address_key')
    geo_hw_addr = geo_hw_addr[['address_key', 'highway_distance_km']].drop_duplicates('address_key')

    fallback = feat.loc[missing_mask, ['address_key']].merge(geo_acc_addr, on='address_key', how='left').merge(geo_hw_addr, on='address_key', how='left')
    for c in ['lat', 'lng', 'nearest_mrt_km', 'road_noise_score', 'facing_road_noise_proxy', 'highway_distance_km']:
        if c in fallback.columns:
            feat.loc[missing_mask, c] = feat.loc[missing_mask, c].fillna(fallback[c].values)

# Factor 5: MRT distance (meters)
feat['dist_to_mrt_m'] = pd.to_numeric(feat.get('nearest_mrt_km'), errors='coerce') * 1000

# Factor 6: orientation score proxy (facing road: penalty)
frp = pd.to_numeric(feat.get('facing_road_noise_proxy'), errors='coerce')
rns = pd.to_numeric(feat.get('road_noise_score'), errors='coerce')
feat['facing_road_flag'] = np.where(frp.notna(), (frp > 0.5).astype(int), np.nan)
feat.loc[feat['facing_road_flag'].isna(), 'facing_road_flag'] = np.where(rns.notna(), (rns > rns.median()).astype(int), np.nan)
feat['orientation_score'] = np.where(feat['facing_road_flag'] == 1, -1.0, 1.0)

# Factor 7: distance to highway (meters)
feat['dist_to_highway_m'] = pd.to_numeric(feat.get('highway_distance_km'), errors='coerce') * 1000

print('Merged feature base rows:', len(feat))
print('Matched coordinates:', int(feat['lat'].notna().sum()), '/', len(feat))
feat[['address_key', 'lat', 'lng', 'dist_to_mrt_m', 'orientation_score', 'dist_to_highway_m']].head()

Merged feature base rows: 263004
Matched coordinates: 263004 / 263004


,address_key,lat,lng,dist_to_mrt_m,orientation_score,dist_to_highway_m
0,174 ANG MO KIO AVE 4,1.375097,103.837619,1176.154764,1.0,605.995848
1,541 ANG MO KIO AVE 10,1.373922,103.855621,2557.672559,1.0,920.428645
2,163 ANG MO KIO AVE 4,1.373549,103.838176,1356.874106,1.0,745.459640
3,446 ANG MO KIO AVE 10,1.367761,103.855357,2904.319336,1.0,1485.511064
4,557 ANG MO KIO AVE 10,1.371626,103.857736,2891.175626,1.0,1267.533033


In [5]:
# Collect or load foodcourt and mall POI data when missing in raw layer
DATASTORE_ENDPOINT = 'https://data.gov.sg/api/action/datastore_search'
ONEMAP_SEARCH_ENDPOINT = 'https://www.onemap.gov.sg/api/common/elastic/search'
NEA_HAWKER_RESOURCE_ID = 'd_4a086da0a5553be1d89383cd90d07ecd'

session = requests.Session()

def fetch_datagov_resource(resource_id: str, limit: int = 5000) -> pd.DataFrame:
    offset = 0
    rows = []
    while True:
        try:
            resp = session.get(DATASTORE_ENDPOINT, params={'resource_id': resource_id, 'limit': limit, 'offset': offset}, timeout=30)
            payload = resp.json()
        except Exception:
            break
        if not payload.get('success'):
            break
        batch = payload['result'].get('records', [])
        rows.extend(batch)
        if len(batch) < limit:
            break
        offset += limit
    return pd.DataFrame(rows)

def onemap_search_all(search_val: str, max_pages: int = 8) -> pd.DataFrame:
    rows = []
    for page in range(1, max_pages + 1):
        try:
            r = session.get(ONEMAP_SEARCH_ENDPOINT, params={
                'searchVal': search_val,
                'returnGeom': 'Y',
                'getAddrDetails': 'Y',
                'pageNum': page,
            }, timeout=30)
            p = r.json()
        except Exception:
            break
        batch = p.get('results', [])
        if not batch:
            break
        rows.extend(batch)
    return pd.DataFrame(rows)

hawker_fp = latest_file_by_pattern(GEO_DIR, 'nea_hawker_centres_*.csv')
if hawker_fp is None:
    hawker = fetch_datagov_resource(NEA_HAWKER_RESOURCE_ID)
    if len(hawker):
        save_fp = GEO_DIR / f'nea_hawker_centres_{RUN_DATE}.csv'
        hawker.to_csv(save_fp, index=False)
        hawker_fp = save_fp
    else:
        # Fallback via OneMap keyword search
        hawker = onemap_search_all('HAWKER CENTRE', max_pages=12)
        if len(hawker):
            hawker = hawker.rename(columns={'SEARCHVAL': 'name', 'LATITUDE': 'lat', 'LONGITUDE': 'lng', 'ADDRESS': 'address'})
            save_fp = GEO_DIR / f'nea_hawker_centres_{RUN_DATE}.csv'
            hawker.to_csv(save_fp, index=False)
            hawker_fp = save_fp

mall_fp = latest_file_by_pattern(GEO_DIR, 'onemap_mall_nodes_*.csv')
if mall_fp is None:
    mall = onemap_search_all('SHOPPING MALL', max_pages=10)
    if len(mall):
        mall = mall.rename(columns={'SEARCHVAL': 'name', 'LATITUDE': 'lat', 'LONGITUDE': 'lng', 'ADDRESS': 'address'})
        save_fp = GEO_DIR / f'onemap_mall_nodes_{RUN_DATE}.csv'
        mall.to_csv(save_fp, index=False)
        mall_fp = save_fp

print('hawker file:', hawker_fp)
print('mall file:', mall_fp)

hawker file: /Users/lorenzolou/VScode/PropertyLens/01_data_layer/raw/google_geo/nea_hawker_centres_20260412.csv
mall file: /Users/lorenzolou/VScode/PropertyLens/01_data_layer/raw/google_geo/onemap_mall_nodes_20260412.csv


In [6]:
def _to_numeric_lat_lng(df: pd.DataFrame, lat_candidates: list[str], lng_candidates: list[str]) -> pd.DataFrame:
    out = df.copy()
    lat_col = next((c for c in lat_candidates if c in out.columns), None)
    lng_col = next((c for c in lng_candidates if c in out.columns), None)
    if lat_col is None or lng_col is None:
        return pd.DataFrame(columns=['lat', 'lng'])
    out['lat'] = pd.to_numeric(out[lat_col], errors='coerce')
    out['lng'] = pd.to_numeric(out[lng_col], errors='coerce')
    return out.dropna(subset=['lat', 'lng'])

def build_balltree(points_lat_lng: np.ndarray) -> BallTree:
    rad = np.radians(points_lat_lng.astype(float))
    return BallTree(rad, metric='haversine')

def nearest_distance_and_count(home_lat_lng: np.ndarray, poi_lat_lng: np.ndarray, radius_km: float) -> tuple[np.ndarray, np.ndarray]:
    if len(home_lat_lng) == 0 or len(poi_lat_lng) == 0:
        n = len(home_lat_lng)
        return np.full(n, np.nan), np.zeros(n, dtype=int)

    home_rad = np.radians(home_lat_lng.astype(float))
    poi_rad = np.radians(poi_lat_lng.astype(float))
    tree = BallTree(poi_rad, metric='haversine')

    dist_rad, _ = tree.query(home_rad, k=1)
    nearest_km = dist_rad[:, 0] * 6371.0

    idx = tree.query_radius(home_rad, r=radius_km / 6371.0)
    count = np.array([len(i) for i in idx], dtype=int)
    return nearest_km, count


home = feat[['lat', 'lng']].copy()
home['lat'] = pd.to_numeric(home['lat'], errors='coerce')
home['lng'] = pd.to_numeric(home['lng'], errors='coerce')
home_valid_mask = home['lat'].notna() & home['lng'].notna()
home_coords = home.loc[home_valid_mask, ['lat', 'lng']].to_numpy()

# Factor 8: foodcourt distance
if hawker_fp is not None and Path(hawker_fp).exists():
    hawker_df = pd.read_csv(hawker_fp)
else:
    hawker_df = pd.DataFrame()
hawker_geo = _to_numeric_lat_lng(hawker_df, ['lat', 'latitude', 'LATITUDE'], ['lng', 'longitude', 'LONGITUDE', 'longtitude'])
hawker_coords = hawker_geo[['lat', 'lng']].drop_duplicates().to_numpy() if len(hawker_geo) else np.empty((0, 2))

food_nearest_km, _ = nearest_distance_and_count(home_coords, hawker_coords, radius_km=1.0)
feat['dist_to_foodcourt_m'] = np.nan
feat.loc[home_valid_mask, 'dist_to_foodcourt_m'] = food_nearest_km * 1000

# Factor 9: nearby commercial (mall) quality/quantity
if mall_fp is not None and Path(mall_fp).exists():
    mall_df = pd.read_csv(mall_fp)
else:
    mall_df = pd.DataFrame()
mall_geo = _to_numeric_lat_lng(mall_df, ['lat', 'LATITUDE'], ['lng', 'LONGITUDE'])
mall_geo['name'] = mall_df.get('name', mall_df.get('SEARCHVAL', pd.Series(index=mall_geo.index, dtype='object'))).astype(str) if len(mall_geo) else ''
mall_geo = mall_geo.drop_duplicates(subset=['lat', 'lng'])
mall_coords = mall_geo[['lat', 'lng']].to_numpy() if len(mall_geo) else np.empty((0, 2))

mall_nearest_km, mall_count_3km = nearest_distance_and_count(home_coords, mall_coords, radius_km=3.0)
feat['dist_to_nearest_mall_m'] = np.nan
feat['mall_count_3km'] = 0
feat.loc[home_valid_mask, 'dist_to_nearest_mall_m'] = mall_nearest_km * 1000
feat.loc[home_valid_mask, 'mall_count_3km'] = mall_count_3km

# Size proxy: brand keyword upweight, then aggregate weighted accessibility within 3km
if len(mall_geo) and len(home_coords):
    names = mall_geo.get('name', pd.Series('', index=mall_geo.index)).astype(str).str.upper()
    big_kw = names.str.contains('MEGA|HUB|CITY|JUNCTION|POINT|PLAZA|CENTRE', regex=True, na=False)
    mall_weight = np.where(big_kw, 1.5, 1.0)

    home_rad = np.radians(home_coords)
    mall_rad = np.radians(mall_coords.astype(float))
    tree = BallTree(mall_rad, metric='haversine')
    neighbors = tree.query_radius(home_rad, r=3.0 / 6371.0, return_distance=True, sort_results=True)

    weighted_access = []
    for idx_arr, dist_arr in zip(neighbors[0], neighbors[1]):
        if len(idx_arr) == 0:
            weighted_access.append(0.0)
            continue
        d_km = np.maximum(dist_arr * 6371.0, 0.05)
        score = np.sum(mall_weight[idx_arr] / (d_km + 0.25))
        weighted_access.append(float(score))

    feat['mall_weighted_access_3km'] = 0.0
    feat.loc[home_valid_mask, 'mall_weighted_access_3km'] = weighted_access
else:
    feat['mall_weighted_access_3km'] = 0.0

print('Hawker points:', len(hawker_coords), '| Mall points:', len(mall_coords))

Hawker points: 61 | Mall points: 158


In [7]:
# Factor 10 & 11: school proximity + primary school implicit quality tier
moe_fp = latest_file_by_pattern(SCHOOL_DIR, 'moe_general_information_of_schools_*.csv')
sg_fp = latest_file_by_pattern(SCHOOL_DIR, 'sgschooling_2015plus_*.csv')

if moe_fp is None or sg_fp is None:
    raise FileNotFoundError('Missing MOE or sgschooling school datasets in raw/schools')

moe = pd.read_csv(moe_fp)
sg = pd.read_csv(sg_fp)

def _norm_apostrophes(s):
    """Normalize curly apostrophes to straight for consistent name matching."""
    return s.replace('\u2019', "'").replace('\u2018', "'")


def build_primary_quality(df_sg: pd.DataFrame) -> pd.DataFrame:
    """Compute school quality from Phase 2B/2C competition ratios.

    The raw CSV has a 4-row structure per school per year:
      school name row, '↳ Vacancy (N)' row, '↳ Applied' row, '↳ Taken' row.
    The Vacancy and Applied rows contain phase-specific slot/applicant counts
    in the 2b, 2c, 2c(s) columns.  We compute:
      competition_ratio = applied / vacancy  (per phase, where vacancy > 0)
    take the max across phases 2B, 2C, 2C(s) as the annual score, then
    average over 2020-2025 for stability.

    This replaces the earlier approach that relied on competition_ratio_extracted
    (98 % null in the source CSV), which collapsed every school to the same
    default value and locked score_school_quality at 5.0.
    """
    df = df_sg.copy()
    df['school'] = df['school'].astype(str).str.strip()
    df = df[df['school'].str.len() > 0].copy()

    is_school  = ~df['school'].str.startswith('↳', na=False)
    is_vacancy =  df['school'].str.startswith('↳ Vacancy', na=False)
    is_applied =  df['school'].str.startswith('↳ Applied', na=False)

    # Forward-fill school name into all child rows
    df['school_name'] = df['school'].where(is_school, np.nan)
    df['school_name'] = df['school_name'].ffill()
    df['school_upper'] = df['school_name'].apply(lambda x: _norm_apostrophes(str(x)).upper())

    # Convert phase columns to numeric (handles '-', '34 PR<1', etc.)
    for col in ['2b', '2c', '2c(s)']:
        if col in df.columns:
            df[col + '_num'] = pd.to_numeric(
                df[col].astype(str).str.extract(r'^(\d+(?:\.\d+)?)')[0],
                errors='coerce'
            )

    # Extract vacancy and applied sub-tables
    _vac_cols = ['school_upper', 'year_int', '2b_num', '2c_num', '2c(s)_num']
    vac = df.loc[is_vacancy, _vac_cols].copy()
    app = df.loc[is_applied, _vac_cols].copy()
    vac.columns = ['school_upper', 'year_int', 'vac_2b', 'vac_2c', 'vac_2cs']
    app.columns = ['school_upper', 'year_int', 'app_2b', 'app_2c', 'app_2cs']

    pa = vac.merge(app, on=['school_upper', 'year_int'], how='inner')

    # Competition ratio per phase; only where vacancy > 0
    for phase in ['2b', '2c', '2cs']:
        v, a = f'vac_{phase}', f'app_{phase}'
        has_v = pa[v].fillna(0) > 0
        pa[f'ratio_{phase}'] = np.where(
            has_v,
            (pa[a].fillna(0) / pa[v].replace(0, np.nan)).clip(upper=5.0),
            np.nan
        )

    pa['max_ratio'] = pa[['ratio_2b', 'ratio_2c', 'ratio_2cs']].max(axis=1)

    # Average over recent years (2020-2025) for stability
    recent = pa[pa['year_int'] >= 2020]
    if recent['school_upper'].nunique() < 10:
        recent = pa  # fall back to all years if recent window too narrow

    quality = recent.groupby('school_upper', as_index=False)['max_ratio'].mean()
    quality.columns = ['school_upper', 'competition_score']

    qmin = quality['competition_score'].min()
    qmax = quality['competition_score'].max()
    quality['school_quality_score'] = 100.0 * (
        (quality['competition_score'] - qmin) / (qmax - qmin + 1e-9)
    )
    quality['school_tier'] = pd.cut(
        quality['school_quality_score'],
        bins=[-1, 40, 70, 100],
        labels=['C', 'B', 'A']
    )
    return quality

primary_quality = build_primary_quality(sg)
_sg_names_set = set(primary_quality['school_upper'])

def _normalize_moe_name(name):
    """Strip 'PRIMARY SCHOOL'/'SCHOOL' suffixes to match sgschooling short names."""
    n = _norm_apostrophes(name.strip().upper())
    for suffix in [' PRIMARY SCHOOL', ' SCHOOL (PRIMARY)', ' SCHOOL (JUNIOR)',
                   ' SCHOOL', ' PRIMARY', ' (PRIMARY)', ' (JUNIOR)']:
        if n.endswith(suffix):
            candidate = n[:-len(suffix)].strip()
            if candidate in _sg_names_set:
                return candidate
    # mid-word SCHOOL removal (e.g. 'ANGLO-CHINESE SCHOOL (JUNIOR)')
    n2 = n.replace(' SCHOOL ', ' ').strip()
    if n2 in _sg_names_set:
        return n2
    # best-effort: strip suffix without existence check
    for suffix in [' PRIMARY SCHOOL', ' SCHOOL (PRIMARY)', ' SCHOOL (JUNIOR)', ' SCHOOL', ' PRIMARY']:
        if n.endswith(suffix):
            return n[:-len(suffix)].strip()
    return n

# Primary school set from MOE
moe['school_name_upper'] = moe['school_name'].apply(
    lambda x: _norm_apostrophes(str(x).strip().upper())
)
moe['school_short'] = moe['school_name'].apply(_normalize_moe_name)
primary = moe[moe['mainlevel_code'].astype(str).str.upper() == 'PRIMARY'].copy()
primary = primary.merge(primary_quality, left_on='school_short', right_on='school_upper', how='left')

# Geocode all schools (cache in raw/google_geo)
school_geo_fp = latest_file_by_pattern(GEO_DIR, 'moe_school_geocode_*.csv')
if school_geo_fp is not None:
    school_geo_raw = pd.read_csv(school_geo_fp)
else:
    rows = []
    endpoint = 'https://www.onemap.gov.sg/api/common/elastic/search'
    for _, r in moe[['school_name', 'address']].drop_duplicates().iterrows():
        query = str(r['address']) if pd.notna(r['address']) and str(r['address']).strip() else str(r['school_name'])
        try:
            resp = session.get(endpoint, params={
                'searchVal': query,
                'returnGeom': 'Y',
                'getAddrDetails': 'Y',
                'pageNum': 1,
            }, timeout=25).json()
            result = resp.get('results', [])
            if result:
                rows.append({
                    'school_name': r['school_name'],
                    'address': r['address'],
                    'lat': result[0].get('LATITUDE'),
                    'lng': result[0].get('LONGITUDE'),
                })
        except Exception:
            continue

    school_geo_raw = pd.DataFrame(rows)
    if len(school_geo_raw):
        school_geo_fp = GEO_DIR / f'moe_school_geocode_{RUN_DATE}.csv'
        school_geo_raw.to_csv(school_geo_fp, index=False)

if len(school_geo_raw) == 0:
    school_geo = pd.DataFrame(columns=['school_name', 'lat', 'lng'])
else:
    school_geo = school_geo_raw.copy()
    school_geo['lat'] = pd.to_numeric(school_geo['lat'], errors='coerce')
    school_geo['lng'] = pd.to_numeric(school_geo['lng'], errors='coerce')
    school_geo = school_geo.dropna(subset=['lat', 'lng']).drop_duplicates(subset=['school_name'])

# All-school proximity
all_school_coords = school_geo[['lat', 'lng']].drop_duplicates().to_numpy() if len(school_geo) else np.empty((0, 2))
all_school_nearest_km, all_school_count_1km = nearest_distance_and_count(home_coords, all_school_coords, radius_km=1.0)
feat['dist_to_nearest_school_m'] = np.nan
feat['school_count_1km'] = 0
feat.loc[home_valid_mask, 'dist_to_nearest_school_m'] = all_school_nearest_km * 1000
feat.loc[home_valid_mask, 'school_count_1km'] = all_school_count_1km

# Primary-tier features within 1km
primary_geo = primary[['school_name', 'school_quality_score']].merge(
    school_geo[['school_name', 'lat', 'lng']],
    on='school_name',
    how='left',
)
primary_geo['lat'] = pd.to_numeric(primary_geo['lat'], errors='coerce')
primary_geo['lng'] = pd.to_numeric(primary_geo['lng'], errors='coerce')
primary_geo = primary_geo.dropna(subset=['lat', 'lng'])

if len(primary_geo) and len(home_coords):
    pq = primary_geo['school_quality_score'].fillna(primary_geo['school_quality_score'].median() if primary_geo['school_quality_score'].notna().any() else 50.0).to_numpy()
    pcoords = primary_geo[['lat', 'lng']].to_numpy()

    home_rad = np.radians(home_coords)
    p_rad = np.radians(pcoords)
    tree = BallTree(p_rad, metric='haversine')
    idx, dist = tree.query_radius(home_rad, r=1.0 / 6371.0, return_distance=True, sort_results=True)

    q_mean = []
    q_top = []
    q_cnt = []
    for ids, d in zip(idx, dist):
        if len(ids) == 0:
            q_mean.append(np.nan)
            q_top.append(np.nan)
            q_cnt.append(0)
            continue
        w = 1.0 / np.maximum(d * 6371.0, 0.05)
        q = pq[ids]
        q_mean.append(float(np.average(q, weights=w)))
        q_top.append(float(np.max(q)))
        q_cnt.append(int(len(ids)))

    feat['primary_school_quality_1km_weighted'] = np.nan
    feat['primary_school_top_quality_1km'] = np.nan
    feat['primary_school_count_1km'] = 0
    feat.loc[home_valid_mask, 'primary_school_quality_1km_weighted'] = q_mean
    feat.loc[home_valid_mask, 'primary_school_top_quality_1km'] = q_top
    feat.loc[home_valid_mask, 'primary_school_count_1km'] = q_cnt
else:
    feat['primary_school_quality_1km_weighted'] = np.nan
    feat['primary_school_top_quality_1km'] = np.nan
    feat['primary_school_count_1km'] = 0

print('Schools geocoded file:', school_geo_fp)
print('All-school points:', len(all_school_coords))
print('Primary schools with coords:', len(primary_geo))

Schools geocoded file: /Users/lorenzolou/VScode/PropertyLens/01_data_layer/raw/google_geo/moe_school_geocode_20260412.csv
All-school points: 334
Primary schools with coords: 179


In [8]:
print(f"\n⚠️ DEBUG: Row count check before school processing")
print(f"   feat rows before school merge: {len(feat)}")
print(f"   primary_geo rows: {len(primary_geo)}")

# KEY FIX: Ensure school data is aggregated and merged cleanly
# The school quality needs to be aggregated by school_name to avoid cartesian products
school_geo_clean = school_geo.drop_duplicates(subset=['school_name']).copy()
primary_clean = primary[['school_name', 'school_quality_score']].drop_duplicates(subset=['school_name']).copy()

print(f"   school_geo deduplicated: {len(school_geo)} -> {len(school_geo_clean)}")
print(f"   primary deduplicated: {len(primary)} -> {len(primary_clean)}")


⚠️ DEBUG: Row count check before school processing
   feat rows before school merge: 263004
   primary_geo rows: 179
   school_geo deduplicated: 336 -> 336
   primary deduplicated: 179 -> 179


In [9]:

# ============================================================================
# FACTOR 12: Business & Social Facility Activity (national-level, by year)
# Source: SoldandRentedHDBPropertiesandFacilities/
#
# These are Singapore-wide annual aggregates of HDB commercial, social, and
# residential unit activity. They capture the macro-economic and community
# infrastructure conditions at the time of each transaction.
#
# Features (all joined on transaction_year):
#   biz_shops_eating_rented      — HDB shops + eating houses rented nationally
#   biz_offices_rented           — HDB offices rented nationally
#   biz_commercial_rented_total  — Total HDB commercial units rented nationally
#   biz_childcare_rented         — Childcare centres rented (family amenity signal)
#   biz_eldercare_rented         — Senior-oriented facilities rented
#   biz_community_rented         — RC Centres + Community Centres rented
#   biz_social_rented_total      — All social/communal facilities rented
#   market_flats_sold_national   — Total HDB flats sold (demand signal)
#   market_flats_rented_national — Total HDB flats rented (rental market signal)
#
# Note: 2025/2026 values are forward-filled from 2024 (latest available data).
# ============================================================================

print("="*70)
print("FACTOR 12: Building Business & Social Activity Features")
print("="*70)

comm_fp   = BIZ_DIR / 'Number of Sold and Rented HDB Commercial Properties.csv'
social_fp = BIZ_DIR / 'Number of Sold and Rented HDB Social Communal Facilities.csv'
resid_fp  = BIZ_DIR / 'Number of Sold and Rented HDB Residential Units.csv'

for fp in [comm_fp, social_fp, resid_fp]:
    if not fp.exists():
        raise FileNotFoundError(f"Missing required biz dataset: {fp}")

comm_raw   = pd.read_csv(comm_fp)
social_raw = pd.read_csv(social_fp)
resid_raw  = pd.read_csv(resid_fp)

# -- Commercial (rented only = current operational stock) --
c = comm_raw[comm_raw['category'] == 'Rented'].copy()

biz_shops_eating = (
    c[c['property_type'].isin(['Shops and Eating Houses', 'Shops', 'Eating Houses'])]
    .groupby('financial_year')['no_of_units'].sum()
    .rename('biz_shops_eating_rented')
)
biz_offices = (
    c[c['property_type'] == 'Offices']
    .groupby('financial_year')['no_of_units'].sum()
    .rename('biz_offices_rented')
)
biz_commercial_total = (
    c.groupby('financial_year')['no_of_units'].sum()
    .rename('biz_commercial_rented_total')
)

# -- Social / Communal (rented only) --
s = social_raw[social_raw['category'] == 'Rented'].copy()

biz_childcare = (
    s[s['facility_type'] == 'Childcare Centres']
    .groupby('financial_year')['no_of_units'].sum()
    .rename('biz_childcare_rented')
)

eldercare_pattern = (
    r'Senior|Active Ageing|Rehabilitation Day Care|'
    r'Day Activity Centres.*Senior|Day Activity Centres.*Disabled.*Senior'
)
biz_eldercare = (
    s[s['facility_type'].str.contains(eldercare_pattern, case=False, na=False, regex=True)]
    .groupby('financial_year')['no_of_units'].sum()
    .rename('biz_eldercare_rented')
)

community_pattern = r"Residents|Community Centres|Neighbourhood Links"
biz_community = (
    s[s['facility_type'].str.contains(community_pattern, case=False, na=False, regex=True)]
    .groupby('financial_year')['no_of_units'].sum()
    .rename('biz_community_rented')
)

biz_social_total = (
    s.groupby('financial_year')['no_of_units'].sum()
    .rename('biz_social_rented_total')
)

# -- Residential market signals --
market_sold = (
    resid_raw[resid_raw['category'] == 'Sold']
    .groupby('financial_year')['no_of_units'].sum()
    .rename('market_flats_sold_national')
)
market_rented = (
    resid_raw[resid_raw['category'] == 'Rented']
    .groupby('financial_year')['no_of_units'].sum()
    .rename('market_flats_rented_national')
)

# -- Combine into lookup table --
biz_lookup = pd.concat([
    biz_shops_eating, biz_offices, biz_commercial_total,
    biz_childcare, biz_eldercare, biz_community, biz_social_total,
    market_sold, market_rented,
], axis=1).reset_index().rename(columns={'financial_year': 'transaction_year'})

# Forward-fill 2025 and 2026 from 2024 (no HDB data beyond 2024 yet)
row_2024 = biz_lookup[biz_lookup['transaction_year'] == 2024]
if len(row_2024):
    for yr in [2025, 2026]:
        if yr not in biz_lookup['transaction_year'].values:
            extra = row_2024.copy()
            extra['transaction_year'] = yr
            biz_lookup = pd.concat([biz_lookup, extra], ignore_index=True)

biz_lookup = biz_lookup.sort_values('transaction_year').reset_index(drop=True)
biz_lookup = biz_lookup.fillna(0)  # rare partial-year gaps -> 0

print(f"Biz lookup: {biz_lookup.shape}  (years {int(biz_lookup['transaction_year'].min())}-{int(biz_lookup['transaction_year'].max())})")
print(biz_lookup[biz_lookup['transaction_year'] >= 2015].to_string(index=False))

# -- Join to feat on transaction_year --
biz_cols = [col for col in biz_lookup.columns if col != 'transaction_year']
feat_before = len(feat)

feat = feat.merge(biz_lookup, on='transaction_year', how='left')
assert len(feat) == feat_before, "Row count changed during biz feature merge!"

null_counts = feat[biz_cols].isnull().sum()
if null_counts.sum() > 0:
    print(f"\n  Null values after merge: {null_counts[null_counts > 0].to_dict()}")
    feat[biz_cols] = feat[biz_cols].fillna(0)

print(f"\nBiz features joined. feat shape: {feat.shape}")
print(f"New columns: {biz_cols}")

# Quick variability sanity check
print("\nVariability check:")
for col in biz_cols:
    uc = feat[col].nunique()
    cv = feat[col].std() / feat[col].mean() if feat[col].mean() != 0 else 0
    print(f"  {col}: {uc} unique, CV={cv:.3f}")


# ============================================================================
# FACTOR 13: Transaction Recency & Interaction Features
# ============================================================================
# Captures how old the transaction is and its interaction with key features.
#   years_since_transaction   : fractional years from transaction month to run date
#   years_since_transaction_sq: squared (non-linear time decay)
#   recency_normalized        : years_since / max(years_since)  [0–1 scale]
#   recency_x_mall_access     : years_since * mall_count_3km
#   recency_x_school_quality  : years_since * primary_school_quality_1km_weighted
#     (especially meaningful now that school quality has real variance 0–100)
# ============================================================================

print("\n" + "="*70)
print("FACTOR 13: Recency & Interaction Features")
print("="*70)

_ref_year  = int(RUN_DATE[:4])
_ref_month = int(RUN_DATE[4:6])
REFERENCE_DECIMAL = _ref_year + _ref_month / 12.0

if 'month' in feat.columns:
    # month column is 'YYYY-MM' string from raw HDB data
    _m = feat['month'].astype(str)
    _tx_decimal = _m.str[:4].astype(float) + _m.str[5:7].astype(float) / 12.0
else:
    _tx_decimal = feat['transaction_year'].astype(float) + 0.5  # mid-year fallback

feat['years_since_transaction'] = (REFERENCE_DECIMAL - _tx_decimal).clip(lower=0)
feat['years_since_transaction_sq'] = feat['years_since_transaction'] ** 2

_ymax = feat['years_since_transaction'].max()
feat['recency_normalized'] = feat['years_since_transaction'] / (_ymax if _ymax > 0 else 1.0)

feat['recency_x_mall_access']    = feat['years_since_transaction'] * feat['mall_count_3km']
feat['recency_x_school_quality'] = feat['years_since_transaction'] * feat['primary_school_quality_1km_weighted']

print(f"  Reference decimal year: {REFERENCE_DECIMAL:.3f}")
print(f"  years_since range: {feat['years_since_transaction'].min():.3f} – {feat['years_since_transaction'].max():.3f}")
print(f"  recency_x_school_quality range: {feat['recency_x_school_quality'].min():.2f} – {feat['recency_x_school_quality'].max():.2f}")
print(f"  recency_x_school_quality unique values: {feat['recency_x_school_quality'].nunique():,}")


FACTOR 12: Building Business & Social Activity Features
Biz lookup: (21, 10)  (years 2006-2026)
 transaction_year  biz_shops_eating_rented  biz_offices_rented  biz_commercial_rented_total  biz_childcare_rented  biz_eldercare_rented  biz_community_rented  biz_social_rented_total  market_flats_sold_national  market_flats_rented_national
             2015                    192.0                39.0                        261.0                  40.0                  26.0                   7.0                    128.0                     21621.0                        3319.0
             2016                    274.0                61.0                        370.0                  51.0                  37.0                  29.0                    148.0                     23161.0                        3799.0
             2017                    275.0                63.0                        364.0                  59.0                  46.0                  31.0                    162.

In [10]:

# Final cleanup, categorical encoding, and temporal split export
base_cols = [
    'resale_price',
    'transaction_year',
    'town',
    'flat_type',
    'flat_model',
    'level_mid',
    'lease_remaining_years',
    'floor_area_sqm',
    'room_count',
    'dist_to_mrt_m',
    'orientation_score',
    'dist_to_highway_m',
    'dist_to_foodcourt_m',
    'dist_to_nearest_mall_m',
    'mall_count_3km',
    'mall_weighted_access_3km',
    'dist_to_nearest_school_m',
    'school_count_1km',
    'primary_school_quality_1km_weighted',
    'primary_school_count_1km',
    # Factor 12: Business & Social facility activity (national, by year)
    'biz_shops_eating_rented',
    'biz_offices_rented',
    'biz_commercial_rented_total',
    'biz_childcare_rented',
    'biz_eldercare_rented',
    'biz_community_rented',
    'biz_social_rented_total',
    'market_flats_sold_national',
    'market_flats_rented_national',
    # Factor 13: recency interaction features
    'years_since_transaction',
    'years_since_transaction_sq',
    'recency_normalized',
    'recency_x_mall_access',
    'recency_x_school_quality',
]

use = feat[base_cols + ['address_key']].copy()

# 🔧 FIX: Remove any exact duplicate rows that may have slipped through
print(f"Rows before dedup: {len(use):,}")
use = use.drop_duplicates(subset=base_cols)
print(f"Rows after dedup: {len(use):,} (removed {len(feat) - len(use):,} exact duplicates)")

for c in [
    'level_mid', 'lease_remaining_years', 'floor_area_sqm', 'room_count',
    'dist_to_mrt_m', 'dist_to_highway_m', 'dist_to_foodcourt_m',
    'dist_to_nearest_mall_m', 'mall_count_3km', 'mall_weighted_access_3km',
    'dist_to_nearest_school_m', 'school_count_1km',
    'primary_school_quality_1km_weighted', 'primary_school_count_1km',
]:
    use[c] = pd.to_numeric(use[c], errors='coerce')

# Robust fill
for c in use.columns:
    if c in ['town', 'flat_type', 'flat_model', 'address_key']:
        use[c] = use[c].astype(str).fillna('UNKNOWN')
    elif c != 'resale_price':
        med = use[c].median() if use[c].notna().any() else 0
        use[c] = use[c].fillna(med)

use = use.dropna(subset=['resale_price'])

# Categorical encoding and temporal split export
print(f"\n📊 CATEGORICAL ENCODING & EXPORT")
print(f"   Input rows: {len(use):,}")

# One-hot encode categorical variables
use_model = pd.get_dummies(use, columns=['town', 'flat_type', 'flat_model'], drop_first=False)
print(f"   After one-hot encoding: {len(use_model)} rows, {len(use_model.columns)} columns")

# Temporal split: use transaction_year to split into train/test
split_year = 2023  # Use data before 2023 for training, 2023+ for testing
train = use_model[use_model['transaction_year'] < split_year].copy()
test = use_model[use_model['transaction_year'] >= split_year].copy()

print(f"\n   Train set (year < {split_year}): {len(train):,} rows")
print(f"   Test set (year >= {split_year}): {len(test):,} rows")
print(f"   Columns after encoding: {len(use_model.columns)}")

# Export to CSV
RUN_DATE = datetime.now().strftime('%Y%m%d')

feature_table_fp = OUTPUT_DIR / f'hdb_feature_table_{RUN_DATE}.csv'
feature_train_fp = OUTPUT_DIR / f'hdb_feature_train_{RUN_DATE}.csv'
feature_test_fp = OUTPUT_DIR / f'hdb_feature_test_{RUN_DATE}.csv'

use_model.to_csv(feature_table_fp, index=False)
train.to_csv(feature_train_fp, index=False)
test.to_csv(feature_test_fp, index=False)

print(f"\n✅ EXPORT COMPLETE")
print(f"   Full table: {feature_table_fp.name} ({len(use_model):,} rows)")
print(f"   Training:  {feature_train_fp.name} ({len(train):,} rows)")
print(f"   Testing:   {feature_test_fp.name} ({len(test):,} rows)")

# Metadata
metadata = {
    'export_date': RUN_DATE,
    'total_rows': len(use_model),
    'train_rows': len(train),
    'test_rows': len(test),
    'columns': len(use_model.columns),
    'split_year': split_year,
    'duplicates_removed': len(feat) - len(use),
}
metadata_fp = OUTPUT_DIR / f'feature_metadata_{RUN_DATE}.json'
with open(metadata_fp, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"   Metadata:  {metadata_fp.name}")


Rows before dedup: 263,004


Rows after dedup: 263,004 (removed 0 exact duplicates)



📊 CATEGORICAL ENCODING & EXPORT
   Input rows: 263,004
   After one-hot encoding: 263004 rows, 86 columns

   Train set (year < 2023): 180,195 rows
   Test set (year >= 2023): 82,809 rows
   Columns after encoding: 86



✅ EXPORT COMPLETE
   Full table: hdb_feature_table_20260415.csv (263,004 rows)
   Training:  hdb_feature_train_20260415.csv (180,195 rows)
   Testing:   hdb_feature_test_20260415.csv (82,809 rows)
   Metadata:  feature_metadata_20260415.json


In [11]:

# ============================================================================
# POST-EXPORT QUALITY GATE
# Reads back the just-saved feature table and verifies:
#   1. No zero-variability features (constant columns)
#   2. No empty values in engineered features
#   3. All OHE columns sum to 1 per row
# Raises ValueError if any critical issue is found.
# ============================================================================

print("\n" + "="*70)
print("POST-EXPORT QUALITY GATE")
print("="*70)

saved = pd.read_csv(feature_table_fp)

id_cols_set = {'resale_price', 'address_key', 'transaction_year', 'month_dt'}
ohe_pfxs = ('town_', 'flat_type_', 'flat_model_', 'school_cluster_')
eng_features = [
    c for c in saved.columns
    if c not in id_cols_set and not c.startswith(ohe_pfxs)
]

issues = []

# 1. Empty values
null_issues = []
for col in eng_features:
    n = saved[col].isnull().sum()
    if n > 0:
        null_issues.append(f"{col}: {n:,} nulls ({100*n/len(saved):.2f}%)")
if null_issues:
    for msg in null_issues:
        print(f"  ❌ EMPTY: {msg}")
    issues.extend(null_issues)
else:
    print(f"  ✅ No empty values in {len(eng_features)} engineered features")

# 2. Zero-variability
zero_var_found = []
for col in eng_features:
    series = saved[col].dropna()
    is_num = pd.api.types.is_numeric_dtype(series)
    std_val = float(series.std()) if is_num else None
    if series.nunique() <= 1 or (is_num and std_val is not None and std_val < 1e-9):
        zero_var_found.append(f"{col} (constant={series.unique()[0] if series.nunique()>=1 else 'N/A'})")
if zero_var_found:
    for msg in zero_var_found:
        print(f"  ❌ ZERO-VAR: {msg}")
    issues.extend(zero_var_found)
else:
    print(f"  ✅ All engineered features have variability")

# 3. OHE sanity
for pfx, label in [('town_', 'town'), ('flat_type_', 'flat_type'), ('flat_model_', 'flat_model')]:
    cols = [c for c in saved.columns if c.startswith(pfx)]
    if not cols:
        continue
    bad = (saved[cols].sum(axis=1) != 1).sum()
    if bad > 0:
        msg = f"{label} OHE: {bad:,} rows where sum ≠ 1"
        print(f"  ❌ OHE: {msg}")
        issues.append(msg)
    else:
        print(f"  ✅ {label} OHE valid ({len(cols)} categories)")

print(f"\n{'='*70}")
if issues:
    raise ValueError(
        f"POST-EXPORT QUALITY GATE FAILED — {len(issues)} issue(s):\n" +
        "\n".join(f"  • {i}" for i in issues)
    )
else:
    print(f"✅ QUALITY GATE PASSED — exported feature table is clean")
    print(f"   Rows: {len(saved):,}  |  Columns: {saved.shape[1]}")



POST-EXPORT QUALITY GATE


  ✅ No empty values in 29 engineered features
  ✅ All engineered features have variability
  ✅ town OHE valid (26 categories)
  ✅ flat_type OHE valid (7 categories)
  ✅ flat_model OHE valid (21 categories)

✅ QUALITY GATE PASSED — exported feature table is clean
   Rows: 263,004  |  Columns: 86


In [12]:
print(f"\n⚠️ DUPLICATION DIAGNOSTIC - Before Categorical Encoding")
print(f"   feat rows: {len(feat):,}")
print(f"   Unique addresses in feat: {feat['address_key'].nunique():,}")
print(f"   Avg rows per address: {len(feat) / feat['address_key'].nunique():.1f}")
print(f"   Expected rows (raw data only): {len(hdb):,}")
print(f"   Unexplained multiplier: {len(feat) / len(hdb):.2f}x")

# Check for duplicate rows (exact match on all columns)
exact_dups = len(feat) - len(feat.drop_duplicates())
print(f"   Exact duplicate rows: {exact_dups:,}")

# Show sample addresses with many rows
addr_counts = feat.groupby('address_key').size().sort_values(ascending=False)
print(f"\n   Top 5 addresses by row count:")
for addr, count in addr_counts.head(5).items():
    print(f"     {addr}: {count} rows")



⚠️ DUPLICATION DIAGNOSTIC - Before Categorical Encoding
   feat rows: 263,004
   Unique addresses in feat: 9,710
   Avg rows per address: 27.1
   Expected rows (raw data only): 263,004
   Unexplained multiplier: 1.00x


   Exact duplicate rows: 0

   Top 5 addresses by row count:
     308A PUNGGOL WALK: 173 rows
     308C PUNGGOL WALK: 156 rows
     187 BOON LAY AVE: 149 rows
     310B PUNGGOL WALK: 143 rows
     87 DAWSON RD: 142 rows


In [13]:

# Validation: check mall counts for Serangoon/Lorong Lew Lian after fix
rows = feat[feat['address_key'].str.contains('LORONG LEW LIAN', na=False)]
print(f"Lorong Lew Lian rows: {len(rows)}")
if len(rows):
    print(rows[['address_key','mall_count_3km','dist_to_nearest_mall_m','mall_weighted_access_3km']].head())
print(f"\nOverall mall_count_3km: mean={feat['mall_count_3km'].mean():.1f}, median={feat['mall_count_3km'].median():.0f}")
print(f"Rows with mall_count_3km == 0: {(feat['mall_count_3km']==0).sum()} / {len(feat)}")


Lorong Lew Lian rows: 0

Overall mall_count_3km: mean=9.5, median=7
Rows with mall_count_3km == 0: 56 / 263004


## Download outputs from Hugging Face

Optional: Sync feature outputs from remote repository.

In [14]:
CLEANUP_MODE = False

In [15]:
from huggingface_hub import snapshot_download, HfApi
from pathlib import Path
import shutil
import os

# USER CONTROL: Set to True to enable download from Hugging Face
DOWNLOAD_OUTPUTS_ENABLED = False

if DOWNLOAD_OUTPUTS_ENABLED:
    HF_TOKEN = os.getenv('HF_TOKEN')
    if not HF_TOKEN:
        raise ValueError('❌ HF_TOKEN not found in environment. Please set it in .env file')
    
    REPO_ID = 'PropertyLens/Resealeflats'
    
    ROOT = Path('/Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens')
    FEATURE_DIR = ROOT / '02_feature_layer' / 'training'
    OUTPUT_DIR = FEATURE_DIR / 'outputs'
    
    api = HfApi(token=HF_TOKEN)
    
    try:
        print(f'🔄 Downloading from {REPO_ID} (latest version)...')
        
        # Step 1: Download version manifest to know what to get
        print(f'\n📋 Fetching VERSION_MANIFEST...')
        try:
            manifest_content = api.hf_hub_download(
                repo_id=REPO_ID,
                filename='02_feature_layer/training/VERSION_MANIFEST.json',
                repo_type='dataset',
                token=HF_TOKEN
            )
            with open(manifest_content, 'r') as f:
                manifest = json.load(f)
            print(f'   ✓ Remote version: {manifest.get("version")}')
            print(f'   ✓ Files in version: {len(manifest.get("files", []))}')
        except Exception as e:
            print(f'   ⚠️  No manifest available: {e}')
            manifest = {}
        
        # Step 2: Download specific files from outputs directory
        print(f'\n⬇️  Downloading files...')
        files_to_download = [
            'hdb_feature_table_*.csv',
            'hdb_feature_train_*.csv',
            'hdb_feature_test_*.csv',
            'feature_metadata_*.json'
        ]
        
        success_count = 0
        for pattern in files_to_download:
            try:
                # Download the entire outputs directory to get latest files
                repo_dir = snapshot_download(
                    repo_id=REPO_ID,
                    repo_type='dataset',
                    token=HF_TOKEN,
                    cache_dir='/tmp/hf_cache',
                    allow_patterns=f'02_feature_layer/training/outputs/{pattern}'
                )
                src_dir = Path(repo_dir) / '02_feature_layer' / 'training' / 'outputs'
                
                if src_dir.exists():
                    for src_file in src_dir.glob(pattern):
                        dest = OUTPUT_DIR / src_file.name
                        shutil.copy2(src_file, dest)
                        print(f'   ✓ {src_file.name}')
                        success_count += 1
            except Exception as e:
                print(f'   ⚠️  Could not download {pattern}: {e}')
        
        print(f'\n✅ DOWNLOAD COMPLETE')
        print(f'Downloaded: {success_count} files')
        print(f'Version: {manifest.get("version", "unknown")}')
        print(f'Local directory: {OUTPUT_DIR}')
        
    except Exception as e:
        print(f'✗ Download failed: {e}')
else:
    print('Download skipped (DOWNLOAD_OUTPUTS_ENABLED = False)')

Download skipped (DOWNLOAD_OUTPUTS_ENABLED = False)

/opt/homebrew/anaconda3/envs/nus/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Upload outputs to Hugging Face with Version Control

Smart upload with automatic cleanup, version tracking, and semantic commit messages.

In [16]:
from huggingface_hub import HfApi
from datetime import datetime, timezone
from pathlib import Path
import os
import re

# USER CONTROL: Set to True to enable upload to Hugging Face
UPLOAD_OUTPUTS_ENABLED = False  # Cleanup enabled; new version control removes old files

if UPLOAD_OUTPUTS_ENABLED:
    HF_TOKEN = os.getenv('HF_TOKEN')
    if not HF_TOKEN:
        raise ValueError('❌ HF_TOKEN not found in environment. Please set it in .env file')
    
    REPO_ID = 'PropertyLens/Resealeflats'
    
    ROOT = Path('/Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens')
    FEATURE_DIR = ROOT / '02_feature_layer' / 'training'
    OUTPUT_DIR = FEATURE_DIR / 'outputs'
    
    api = HfApi(token=HF_TOKEN)
    timestamp = datetime.now(timezone.utc).isoformat()
    current_date = datetime.now().strftime('%Y%m%d')
    
    try:
        print(f'🔄 Upload to {REPO_ID} with version control...')
        
        # Step 1: Scan remote repository and delete OLD versions
        print(f'\n🗑️  Cleaning up old remote versions...')
        files_deleted = []
        
        try:
            # Get all files in the remote repository
            repo_files = api.list_files_info(repo_id=REPO_ID, repo_type='dataset', recursive=True)
            
            # Pattern to match dated files (e.g., hdb_feature_table_20260412.csv)
            date_pattern = re.compile(r'_(\d{8})\.')
            
            for file_info in repo_files:
                file_path = file_info.path
                # Skip non-feature layer files and manifests
                if not file_path.startswith('02_feature_layer/'):
                    continue
                if 'VERSION_MANIFEST' in file_path:
                    continue
                
                # Check if file has a date stamp
                match = date_pattern.search(file_path)
                if match:
                    file_date = match.group(1)
                    # Delete if it's NOT the current date
                    if file_date != current_date:
                        try:
                            api.delete_file(
                                path_in_repo=file_path,
                                repo_id=REPO_ID,
                                repo_type='dataset',
                                commit_message=f'🧹 Cleanup: Remove old feature file from {file_date}'
                            )
                            files_deleted.append(file_path.split('/')[-1])
                            print(f'   ✓ Deleted: {file_path.split("/")[-1]}')
                        except Exception as e:
                            print(f'   ⚠️  Could not delete {file_path}: {e}')
            
            if files_deleted:
                print(f'   Removed {len(files_deleted)} old version(s)')
            else:
                print(f'   No old versions found (repository clean)')
                
        except Exception as e:
            print(f'   ⚠️  Could not scan repository: {e}')
        
        # Step 2: Upload new files with version control commit message
        print(f'\n⬆️  Uploading current version ({current_date})...')
        items = sorted(OUTPUT_DIR.glob('*'))  # Get all files in outputs
        success_count = 0
        uploaded_files = []
        
        for item in items:
            if item.is_file():
                path_in_repo = f'02_feature_layer/training/outputs/{item.name}'
                try:
                    # Determine what changed based on file type
                    if 'feature_table' in item.name:
                        file_desc = 'Feature table (full deduplicated dataset)'
                    elif 'feature_train' in item.name:
                        file_desc = 'Training set (year < 2023)'
                    elif 'feature_test' in item.name:
                        file_desc = 'Test set (year >= 2023)'
                    else:
                        file_desc = item.name
                    
                    # Create semantic commit message
                    commit_message = f"""📊 Feature Layer v{current_date}

Update: {file_desc}
- Date: {timestamp}
- Action: Version update with cleanup
- Version control: {len(files_deleted)} old version(s) removed
"""
                    
                    api.upload_file(
                        path_or_fileobj=str(item),
                        repo_id=REPO_ID,
                        path_in_repo=path_in_repo,
                        repo_type='dataset',
                        commit_message=commit_message
                    )
                    print(f'   ✓ {item.name}')
                    success_count += 1
                    uploaded_files.append(item.name)
                except Exception as e:
                    print(f'   ✗ {item.name}: {e}')
        
        # Step 3: Create and upload version manifest
        print(f'\n📄 Creating version manifest...')
        manifest = {
            'version': current_date,
            'timestamp': timestamp,
            'files': uploaded_files,
            'duplicates_removed': metadata.get('duplicates_removed'),
            'total_rows': metadata.get('total_rows'),
            'train_rows': metadata.get('train_rows'),
            'test_rows': metadata.get('test_rows'),
            'split_year': metadata.get('split_year'),
            'old_versions_removed': len(files_deleted)
        }
        
        manifest_fp = OUTPUT_DIR / 'VERSION_MANIFEST.json'
        with open(manifest_fp, 'w') as f:
            json.dump(manifest, f, indent=2)
        
        try:
            api.upload_file(
                path_or_fileobj=str(manifest_fp),
                repo_id=REPO_ID,
                path_in_repo='02_feature_layer/training/VERSION_MANIFEST.json',
                repo_type='dataset',
                commit_message=f'📌 Version manifest v{current_date} (removed {len(files_deleted)} old versions)'
            )
            print(f'   ✓ VERSION_MANIFEST.json uploaded')
        except Exception as e:
            print(f'   ⚠️  Could not upload manifest: {e}')
        
        # Step 4: Summary report
        print(f'\n' + '='*70)
        print(f'✅ UPLOAD COMPLETE (Version {current_date})')
        print(f'='*70)
        print(f'Repository: https://huggingface.co/datasets/{REPO_ID}')
        print(f'Old versions deleted: {len(files_deleted)}')
        print(f'New files uploaded: {success_count}')
        print(f'Manifest: VERSION_MANIFEST.json')
        print(f'='*70)
        
    except Exception as e:
        print(f'✗ Upload failed: {e}')
else:
    print('Upload skipped (UPLOAD_OUTPUTS_ENABLED = False)')

Upload skipped (UPLOAD_OUTPUTS_ENABLED = False)
